# AI-Task3: Automated Reasoning Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jaweriamushtaq41-coder/Smart_District_2/blob/main/AI-Task3/AI_Task3_Reasoning_Pipeline.ipynb)

**Kya banana hai:** Bara business masla -> CoT se sub-questions -> har ek alag hal -> aakhri recommended action. Har step ka output aur reasoning timestamp wali log file mein.

**Concepts:** CoT, prompt chaining, `logging` module

**Colab mein kaise chalayen:**
1. Upar *Open In Colab* dabao (ya ye file Colab mein upload karo).
2. *Runtime > Run all* dabao.
3. Keys ho to Colab ke left side ke **Secrets** (chaabi ka icon) mein daalo. Bina key ke bhi chalta hai (**demo mode**, nakli jawab).

> Notebook ko share karne se pehle apni keys kabhi output / cell mein mat chhodo.

## 1. Libraries install aur keys (`.env`)

In [1]:
!pip -q install -U python-dotenv openai anthropic google-generativeai groq

In [2]:
import os

# .env file banao. Keys do tareeqon se aa sakti hain:
#  1) Colab "Secrets" (left side ka chaabi wala icon), secret ka naam wahi jo neeche list mein hai, "Notebook access" ON
#  2) apni .env file Colab ke "Files" panel (left side ka folder icon) mein drag-drop kar do, run karne se pehle
NAMES = ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GOOGLE_API_KEY", "GROQ_API_KEY"]

existing = {}                       # pehle se .env ho (upload ki hui) to uski keys bacha lo
if os.path.exists(".env"):
    with open(".env") as f:
        for line in f:
            if "=" in line and not line.startswith("#"):
                key, value = line.strip().split("=", 1)
                existing[key] = value

lines = []
for name in NAMES:
    value = existing.get(name, "")
    try:
        from google.colab import userdata
        value = userdata.get(name) or value
    except Exception:          # Colab nahi hai, ya ye secret nahi banaya
        pass
    lines.append(f"{name}={value}")

with open(".env", "w") as f:
    f.write("\n".join(lines) + "\n")

from dotenv import load_dotenv
load_dotenv(".env", override=True)

found = [n for n in NAMES if os.getenv(n)]
print("Keys mili:", found if found else "koi nahi (demo mode chalega)")

Keys mili: koi nahi (demo mode chalega)


## 2. Helper: chaaron providers ek jaisa
`ask(provider, prompt, ...)` har provider ke SDK ko bulata hai aur ek hi shakal ka `Reply` deta hai
(text, time, tokens, cost). `demo=True` par nakli jawab milta hai. Ye cell sirf chalana hai, samajhne ke liye parh lo.

In [3]:
import os
import time
from dataclasses import dataclass

from dotenv import load_dotenv

load_dotenv()   # .env file se keys parh leta hai

PROVIDERS = ["openai", "claude", "gemini", "groq"]

# .env mein model ka naam badal sakte ho (model ke naam jaldi purane ho jate hain)
DEFAULT_MODELS = {
    "openai": os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
    "claude": os.getenv("ANTHROPIC_MODEL", "claude-3-5-haiku-latest"),
    "gemini": os.getenv("GEMINI_MODEL", "gemini-1.5-flash"),
    "groq": os.getenv("GROQ_MODEL", "llama-3.1-8b-instant"),
}

KEY_NAMES = {
    "openai": "OPENAI_API_KEY",
    "claude": "ANTHROPIC_API_KEY",
    "gemini": "GOOGLE_API_KEY",
    "groq": "GROQ_API_KEY",
}

# (input, output) dollar per 1 million tokens. Ye sirf ANDAZA hai, asli rate provider ki site par dekho.
PRICES = {
    "openai": (0.15, 0.60),
    "claude": (0.80, 4.00),
    "gemini": (0.075, 0.30),
    "groq": (0.05, 0.08),
}


# demo mein har provider ki nakli raftaar (Groq sab se tez, jaisa asal mein hota hai)
DEMO_DELAY = {"openai": 0.12, "claude": 0.18, "gemini": 0.09, "groq": 0.03}


class LLMError(Exception):
    """Provider se jawab na aaye ya key / SDK ka masla ho."""


@dataclass
class Reply:
    provider: str
    model: str
    text: str
    seconds: float
    input_tokens: int
    output_tokens: int
    cost_usd: float
    demo: bool = False

    @property
    def total_tokens(self):
        return self.input_tokens + self.output_tokens


def has_key(provider):
    return bool(os.getenv(KEY_NAMES[provider], "").strip())


def available_providers():
    """Jin providers ki key .env mein mojood hai."""
    return [p for p in PROVIDERS if has_key(p)]


def estimate_cost(provider, input_tokens, output_tokens):
    price_in, price_out = PRICES[provider]
    return (input_tokens * price_in + output_tokens * price_out) / 1_000_000


def rough_tokens(text):
    # angrezi mein taqreeban 4 huroof = 1 token (sirf andaza)
    return max(1, len(text) // 4)


# ---------------- har provider ka apna call ----------------
def _call_openai(model, prompt, system, temperature, max_tokens):
    from openai import OpenAI
    client = OpenAI(api_key=os.getenv(KEY_NAMES["openai"]))
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    r = client.chat.completions.create(
        model=model, messages=messages, temperature=temperature, max_tokens=max_tokens)
    return r.choices[0].message.content, r.usage.prompt_tokens, r.usage.completion_tokens


def _call_groq(model, prompt, system, temperature, max_tokens):
    from groq import Groq
    client = Groq(api_key=os.getenv(KEY_NAMES["groq"]))
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    r = client.chat.completions.create(
        model=model, messages=messages, temperature=temperature, max_tokens=max_tokens)
    return r.choices[0].message.content, r.usage.prompt_tokens, r.usage.completion_tokens


def _call_claude(model, prompt, system, temperature, max_tokens):
    import anthropic
    client = anthropic.Anthropic(api_key=os.getenv(KEY_NAMES["claude"]))
    kwargs = {"model": model, "max_tokens": max_tokens, "temperature": temperature,
              "messages": [{"role": "user", "content": prompt}]}
    if system:
        kwargs["system"] = system          # Claude mein system alag parameter hota hai
    r = client.messages.create(**kwargs)
    return r.content[0].text, r.usage.input_tokens, r.usage.output_tokens


def _call_gemini(model, prompt, system, temperature, max_tokens):
    import google.generativeai as genai
    genai.configure(api_key=os.getenv(KEY_NAMES["gemini"]))
    gm = genai.GenerativeModel(model, system_instruction=system)
    r = gm.generate_content(
        prompt, generation_config={"temperature": temperature, "max_output_tokens": max_tokens})
    usage = r.usage_metadata
    return r.text, usage.prompt_token_count, usage.candidates_token_count


CALLERS = {"openai": _call_openai, "claude": _call_claude, "gemini": _call_gemini, "groq": _call_groq}


def _demo_reply(provider, prompt, mock_reply):
    """Nakli jawab: mock_reply (text ya function) ho to wahi, warna ek chhota sa default."""
    if callable(mock_reply):
        text = mock_reply(prompt)
    elif mock_reply is not None:
        text = mock_reply
    else:
        text = (f"[DEMO] Ye {provider} ka nakli jawab hai. Asli jawab ke liye .env mein "
                f"{KEY_NAMES[provider]} likho.")
    return text


def ask(provider, prompt, system=None, temperature=0.7, max_tokens=500,
        demo=False, mock_reply=None, retries=2):
    """Ek provider ko prompt bhejo aur Reply wapas lo (text, time, tokens, cost)."""
    if provider not in CALLERS:
        raise LLMError(f"Unknown provider '{provider}'. Choose from {PROVIDERS}")
    model = DEFAULT_MODELS[provider]
    start = time.perf_counter()

    if demo:
        text = _demo_reply(provider, prompt, mock_reply)
        in_tok = rough_tokens((system or "") + prompt)
        out_tok = rough_tokens(text)
        time.sleep(DEMO_DELAY[provider])         # thora sa waqt lagne ka ehsaas
    else:
        if not has_key(provider):
            raise LLMError(f"{KEY_NAMES[provider]} .env mein nahi mili")
        for attempt in range(retries + 1):
            try:
                text, in_tok, out_tok = CALLERS[provider](model, prompt, system, temperature, max_tokens)
                break
            except ImportError as e:
                raise LLMError(f"{provider} ka SDK install nahi hai: {e}") from e
            except Exception as e:                # SDK ke alag alag errors ek jaisa kar diye
                too_many = "429" in str(e) or "rate" in str(e).lower()
                if too_many and attempt < retries:
                    time.sleep(2 ** attempt)      # 1s, 2s ruk kar dobara
                    continue
                raise LLMError(f"{provider} error: {e}") from e

    seconds = time.perf_counter() - start
    return Reply(provider=provider, model=model, text=text.strip(), seconds=seconds,
                 input_tokens=in_tok, output_tokens=out_tok,
                 cost_usd=estimate_cost(provider, in_tok, out_tok), demo=demo)

## 3. Pipeline ke functions
- `decompose_prompt`: masle ko 3 sawalon mein todta hai (CoT)
- `solve_prompt`: har sawal alag hal
- `synthesise_prompt`: sab jawab mila kar aakhri action
- `run_step`: har step ka prompt, jawab aur tokens log file mein likhta hai

In [4]:
import logging
import os
import re
import sys
from datetime import datetime

In [5]:
DEFAULT_PROBLEM = ("Our online clothing shop's sales dropped 20% this quarter while advertising "
                   "spend stayed the same. What should we do?")

log = logging.getLogger("reasoning")


def setup_logging():
    os.makedirs("logs", exist_ok=True)
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    path = os.path.join("logs", f"reasoning_{stamp}.log")
    log.setLevel(logging.INFO)
    log.handlers.clear()
    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
    for handler in (logging.FileHandler(path, encoding="utf-8"), logging.StreamHandler(sys.stdout)):
        handler.setFormatter(fmt)
        log.addHandler(handler)
    return path


# ---------------- prompts ----------------
def decompose_prompt(problem):
    return (f"You are a business analyst. Think step by step and break this business problem into "
            f"exactly 3 smaller sub-questions that, once answered, would solve it.\n"
            f"Reply ONLY as a numbered list, one sub-question per line.\n\nProblem: {problem}")


def solve_prompt(problem, sub_question):
    return (f"Business problem: {problem}\n\nSub-question: {sub_question}\n\n"
            f"Reason step by step in 2-3 short sentences, then end with a line starting 'Answer:'.")


def synthesise_prompt(problem, solved):
    parts = "\n".join(f"- {q}\n  {a}" for q, a in solved)
    return (f"Synthesise the findings below into ONE recommended action plan (max 5 lines).\n\n"
            f"Business problem: {problem}\n\nFindings:\n{parts}\n\nRecommended action:")


# ---------------- demo (nakli) jawab ----------------
def demo_brain(prompt):
    if "Synthesise" in prompt:
        return ("1. Fix the checkout flow first (biggest and cheapest win).\n"
                "2. Move ad budget to the best-converting channel.\n"
                "3. Run a 2-week discount test on slow items.\n"
                "4. Review the numbers weekly and stop what does not work.")
    if "Sub-question:" in prompt:
        if "traffic" in prompt.split("Sub-question:")[1].lower():
            return ("Visitor numbers look the same as last quarter, so advertising is still bringing people in.\n"
                    "Answer: traffic did not fall, so the problem is after the visitor arrives.")
        if "leave" in prompt.split("Sub-question:")[1].lower():
            return ("Most visitors drop out at the payment page, which suggests a checkout problem.\n"
                    "Answer: the checkout step loses the most buyers.")
        return ("Two ad channels give many clicks but almost no sales, while email still converts well.\n"
                "Answer: some channels waste budget and should be reduced.")
    return ("1. Did website traffic fall, or only the number of buyers?\n"
            "2. Where in the buying steps do visitors leave?\n"
            "3. Which products or ad channels changed in performance?")


def parse_numbered(text):
    """'1. abc' wali lines nikalta hai. Agar format na mile to poora text ek sawal maan leta hai."""
    items = re.findall(r"^\s*\d+[.)]\s*(.+)$", text, flags=re.MULTILINE)
    return [i.strip() for i in items] or [text.strip()]


def run_step(name, provider, prompt, demo):
    log.info("===== %s =====", name)
    log.info("PROMPT:\n%s", prompt)
    reply = ask(provider, prompt, temperature=0.2, max_tokens=400, demo=demo, mock_reply=demo_brain)
    log.info("RESPONSE:\n%s", reply.text)
    log.info("tokens in=%d out=%d | %.2fs | ~$%.6f", reply.input_tokens, reply.output_tokens,
             reply.seconds, reply.cost_usd)
    return reply


def run_pipeline(problem, provider, demo):
    log.info("PROBLEM: %s", problem)

    step1 = run_step("STEP 1: Sub-questions banana (CoT)", provider, decompose_prompt(problem), demo)
    sub_questions = parse_numbered(step1.text)
    log.info("Mile %d sub-questions", len(sub_questions))

    solved = []
    for i, q in enumerate(sub_questions, start=1):
        step = run_step(f"STEP 2.{i}: Sub-question hal karna", provider, solve_prompt(problem, q), demo)
        solved.append((q, step.text))

    final = run_step("STEP 3: Final recommendation", provider, synthesise_prompt(problem, solved), demo)
    return sub_questions, solved, final.text

## 4. Provider chuno

In [6]:
FORCE_DEMO = False      # True likhoge to hamesha nakli jawab
PROVIDER = None         # "groq", "openai", "claude" ya "gemini". None = jis ki key mili
demo = FORCE_DEMO
provider = PROVIDER or (available_providers() or [None])[0]
if provider is None:
    print("[!] Koi API key nahi mili, DEMO mode chal raha hai.")
    demo, provider = True, "groq"
print("provider =", provider, "| demo =", demo)

[!] Koi API key nahi mili, DEMO mode chal raha hai.
provider = groq | demo = True


## 5. Chalao
`PROBLEM` badal kar apna business masla likho.

In [7]:
PROBLEM = DEFAULT_PROBLEM

log_path = setup_logging()
sub_questions, solved, recommendation = run_pipeline(PROBLEM, provider, demo)

print("\n" + "=" * 60)
print("FINAL RECOMMENDED ACTION")
print("=" * 60)
print(recommendation)
print("\nPoori reasoning is log file mein hai:", log_path)

2026-10-31 18:07:40,819 | INFO | PROBLEM: Our online clothing shop's sales dropped 20% this quarter while advertising spend stayed the same. What should we do?
2026-10-31 18:07:40,820 | INFO | ===== STEP 1: Sub-questions banana (CoT) =====
2026-10-31 18:07:40,820 | INFO | PROMPT:
You are a business analyst. Think step by step and break this business problem into exactly 3 smaller sub-questions that, once answered, would solve it.
Reply ONLY as a numbered list, one sub-question per line.

Problem: Our online clothing shop's sales dropped 20% this quarter while advertising spend stayed the same. What should we do?
2026-10-31 18:07:40,850 | INFO | RESPONSE:
1. Did website traffic fall, or only the number of buyers?
2. Where in the buying steps do visitors leave?
3. Which products or ad channels changed in performance?
2026-10-31 18:07:40,850 | INFO | tokens in=84 out=40 | 0.03s | ~$0.000007
2026-10-31 18:07:40,851 | INFO | Mile 3 sub-questions
2026-10-31 18:07:40,851 | INFO | ===== STEP 2

## 6. Log file dekho

In [8]:
print(open(log_path, encoding='utf-8').read()[:1800])

2026-10-31 18:07:40,819 | INFO | PROBLEM: Our online clothing shop's sales dropped 20% this quarter while advertising spend stayed the same. What should we do?
2026-10-31 18:07:40,820 | INFO | ===== STEP 1: Sub-questions banana (CoT) =====
2026-10-31 18:07:40,820 | INFO | PROMPT:
You are a business analyst. Think step by step and break this business problem into exactly 3 smaller sub-questions that, once answered, would solve it.
Reply ONLY as a numbered list, one sub-question per line.

Problem: Our online clothing shop's sales dropped 20% this quarter while advertising spend stayed the same. What should we do?
2026-10-31 18:07:40,850 | INFO | RESPONSE:
1. Did website traffic fall, or only the number of buyers?
2. Where in the buying steps do visitors leave?
3. Which products or ad channels changed in performance?
2026-10-31 18:07:40,850 | INFO | tokens in=84 out=40 | 0.03s | ~$0.000007
2026-10-31 18:07:40,851 | INFO | Mile 3 sub-questions
2026-10-31 18:07:40,851 | INFO | ===== STEP 2

## Kya seekha
- **Prompt chaining**: ek step ka output agle step ka input banta hai.
- Bara masla chhote sawalon mein tootne se jawab behtar aate hain.
- Har step ka log rakhne se pata chalta hai model ne kahan kya socha (debugging aasan).